# Prepare clean paid-purchase history

## Goal

Convert the private sales export into one row per **customer + date + product**.

Only ordinary paid product sales (`ПРОДАЖА`) are retained. Gift rows (`ПОДАРОК`) are removed completely and do not affect labels, quantities, recency, reorder timing, or the product catalogue.

## 1. Setup

In [17]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw_data_dir = project_root / "data" / "raw"

xlsx_raw_file = list(raw_data_dir.glob("*.xlsx"))[0]

if not xlsx_raw_file.exists():
    raise FileNotFoundError(f"Raw data file not found in {raw_data_dir}")
else:
    input_path = xlsx_raw_file

output_path = project_root / "data" / "interim" / "cleaned_purchases.csv"

## 2. Load, rename, and normalize columns

Customer and product codes are loaded as strings so pandas cannot interpret identifiers as numbers. Dates remain datetimes, and quantity remains numeric.

In [18]:
column_mapping = {
    "КлиентДляОплатыКод": "customer_id",
    "ТоварКод": "product_id",
    "Категория": "product_category",
    "БизнесЛиния": "business_line",
    "ДатаПродажи": "purchase_date",
    "Количество": "quantity",
    "Gen_ Bus_ Posting Group": "transaction_type",
    "Gen_ Prod_ Posting Group": "item_type",
}

identifier_dtypes = {
    "КлиентДляОплатыКод": "string",
    "ТоварКод": "string",
}

df = pd.read_excel(input_path, dtype=identifier_dtypes)

missing_columns = sorted(set(column_mapping) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing expected source columns: {missing_columns}")

df = df.rename(columns=column_mapping)

text_columns = [
    "customer_id",
    "product_id",
    "product_category",
    "business_line",
    "transaction_type",
    "item_type",
]

for column in text_columns:
    df[column] = df[column].astype("string").str.strip()

df["purchase_date"] = pd.to_datetime(df["purchase_date"], errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")

print(f"Loaded {len(df):,} source rows.")
df.head(10)

Loaded 113,621 source rows.


,Межфирма,transaction_type,item_type,Компания,Дата,purchase_date,ФинСчет,ДокументДвижения,business_line,product_category,...,АкцияКод,Акция,ОписаниеАкции,УсловияОплаты,ТипОплаты,ТипКлиента,Услуга,УслугаНазвание,АдресДоставки,ILEDocType
0,0.0,ПРОДАЖА,ТОВАР,IG,2026-05-12,2026-05-12,281100,НОВА ПОШТА,GREASE,ИНДУСТР-СМ,...,NaN,NaN,NaN,ПРЕДОПЛАТА,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Карачівське шосе, 44, Харків, Харківська облас...",1.0
1,0.0,ПРОДАЖА,ТОВАР,IG,2026-05-12,2026-05-12,281100,НОВА ПОШТА,GREASE,ИНДУСТР-СМ,...,NaN,NaN,NaN,ПРЕДОПЛАТА,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Карачівське шосе, 44, Харків, Харківська облас...",1.0
2,0.0,ПРОДАЖА,ТОВАР,IG,2026-02-23,2026-02-23,281100,П Р С+А034555,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1.0
3,0.0,ПРОДАЖА,ТОВАР,IG,2026-02-23,2026-02-23,281100,П Р С+А034555,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1.0
4,0.0,ПРОДАЖА,ТОВАР,IG,2026-03-18,2026-03-18,281100,П РС+А 034555,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1.0
5,0.0,ПРОДАЖА,ТОВАР,IG,2026-02-10,2026-02-10,281100,П РС+А034555,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1.0
6,0.0,ПРОДАЖА,ТОВАР,IG,2026-02-10,2026-02-10,281100,П РС+А034555,GREASE,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1.0
7,0.0,ПРОДАЖА,ТОВАР,IG,2026-02-10,2026-02-10,902100,П РС+А034555,GREASE,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1.0
8,0.0,ПРОДАЖА,ТОВАР,IG,2024-12-26,2024-12-26,281100,П Р С +У 9 4 3 5 04,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН21,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1.0
9,0.0,ПРОДАЖА,ТОВАР,IG,2024-12-13,2024-12-13,281100,П Р С + У943504,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН21,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1.0


## 3. Keep positive paid product purchases

A retained row must identify a customer, date, and product; represent an actual product (`ТОВАР`); be an ordinary sale (`ПРОДАЖА`); and have a positive quantity. Gift rows and every other transaction type are excluded before aggregation.

In [19]:
purchase_mask = (
    df["customer_id"].notna()
    & df["product_id"].notna()
    & df["purchase_date"].notna()
    & df["item_type"].eq("ТОВАР")
    & df["transaction_type"].eq("ПРОДАЖА")
    & df["quantity"].gt(0)
)

product_purchases = df.loc[purchase_mask].copy()

gift_product_rows = int(
    (
        df["item_type"].eq("ТОВАР")
        & df["transaction_type"].eq("ПОДАРОК")
        & df["quantity"].gt(0)
    ).sum()
)
assert product_purchases["transaction_type"].eq("ПРОДАЖА").all()
print(f"Excluded {gift_product_rows:,} positive gift-product rows.")
product_purchases.head(10)

Excluded 1,954 positive gift-product rows.


,Межфирма,transaction_type,item_type,Компания,Дата,purchase_date,ФинСчет,ДокументДвижения,business_line,product_category,...,АкцияКод,Акция,ОписаниеАкции,УсловияОплаты,ТипОплаты,ТипКлиента,Услуга,УслугаНазвание,АдресДоставки,ILEDocType
0,0.0,ПРОДАЖА,ТОВАР,IG,2026-05-12,2026-05-12,281100,НОВА ПОШТА,GREASE,ИНДУСТР-СМ,...,NaN,NaN,NaN,ПРЕДОПЛАТА,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Карачівське шосе, 44, Харків, Харківська облас...",1.0
1,0.0,ПРОДАЖА,ТОВАР,IG,2026-05-12,2026-05-12,281100,НОВА ПОШТА,GREASE,ИНДУСТР-СМ,...,NaN,NaN,NaN,ПРЕДОПЛАТА,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Карачівське шосе, 44, Харків, Харківська облас...",1.0
2,0.0,ПРОДАЖА,ТОВАР,IG,2026-02-23,2026-02-23,281100,П Р С+А034555,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1.0
3,0.0,ПРОДАЖА,ТОВАР,IG,2026-02-23,2026-02-23,281100,П Р С+А034555,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1.0
4,0.0,ПРОДАЖА,ТОВАР,IG,2026-03-18,2026-03-18,281100,П РС+А 034555,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1.0
5,0.0,ПРОДАЖА,ТОВАР,IG,2026-02-10,2026-02-10,281100,П РС+А034555,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1.0
6,0.0,ПРОДАЖА,ТОВАР,IG,2026-02-10,2026-02-10,281100,П РС+А034555,GREASE,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН30,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1.0
8,0.0,ПРОДАЖА,ТОВАР,IG,2024-12-26,2024-12-26,281100,П Р С +У 9 4 3 5 04,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН21,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1.0
9,0.0,ПРОДАЖА,ТОВАР,IG,2024-12-13,2024-12-13,281100,П Р С + У943504,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН21,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1.0
10,0.0,ПРОДАЖА,ТОВАР,IG,2024-11-29,2024-11-29,281100,П Р С +У943504,INDUSTRY,ИНДУСТР-СМ,...,NaN,NaN,NaN,ДН21,БЕЗНАЛ,НАЦИОНАЛЬН,NaN,NaN,"Відділення №1: вул. Роменська, 13, смт Липова",1.0


## 4. Combine duplicate customer-date-product rows

The grouping key is `customer_id + purchase_date + product_id`. Different products on the same date remain separate. Repeated rows for the same product are combined.

Purchase quantities are summed. `first` keeps the first non-missing category and business line.

In [20]:
grouping_columns = ["customer_id", "purchase_date", "product_id"]

cleaned_purchases = (
    product_purchases
    .groupby(
        grouping_columns,
        as_index=False,
        sort=False,
    )
    .agg(
        quantity=("quantity", "sum"),
        product_category=("product_category", "first"),
        business_line=("business_line", "first"),
    )
    .sort_values(grouping_columns)
    .reset_index(drop=True)
)

cleaned_purchases.head(10)

,customer_id,purchase_date,product_id,quantity,product_category,business_line
0,КЛН-С003715,2024-03-22,ТОВ-У513202,180.0,ИНДУСТР-СМ,SKY_GREASE
1,КЛН-С003715,2024-05-27,ТОВ-У513454,970.0,АВТО-СМ,CRT
2,КЛН-С003715,2024-05-28,ТОВ-У501685,360.0,ИНДУСТР-СМ,GREASE
3,КЛН-С003715,2024-05-28,ТОВ-У503091,209.0,АВТО-СМ,CRT
4,КЛН-С003715,2024-05-28,ТОВ-У512994,1000.0,ИНДУСТР-СМ,SKY_MWO
5,КЛН-С003715,2024-05-28,ТОВ-У513202,180.0,ИНДУСТР-СМ,SKY_GREASE
6,КЛН-С003715,2024-06-06,ТОВ-У500734,18.0,ИНДУСТР-СМ,GREASE
7,КЛН-С003715,2024-07-19,ТОВ-У002665,4.0,АВТО-СМ,PCMO
8,КЛН-С003715,2024-07-19,ТОВ-У500636,209.0,ИНДУСТР-СМ,INDUSTRY
9,КЛН-С003715,2024-09-20,ТОВ-У501685,360.0,ИНДУСТР-СМ,GREASE


## 5. Validate and save

The checks stop the notebook if duplicate keys, non-positive purchases, gift-related fields, or missing numeric values survive. The validated table is then saved as CSV with dates written as `YYYY-MM-DD`.

In [21]:
duplicate_count = int(cleaned_purchases.duplicated(grouping_columns).sum())
non_positive_purchase_count = int(cleaned_purchases["quantity"].le(0).sum())

assert duplicate_count == 0, f"Found {duplicate_count} duplicate grouping keys."
assert non_positive_purchase_count == 0, (
    f"Found {non_positive_purchase_count} non-positive purchases."
)
assert cleaned_purchases.select_dtypes(include="number").notna().all().all()
assert {
    "gift_quantity",
    "received_quantity",
    "paid_quantity",
}.isdisjoint(cleaned_purchases.columns)

output_path.parent.mkdir(parents=True, exist_ok=True)
cleaned_purchases.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%d",
)

saved_purchases = pd.read_csv(
    output_path,
    dtype={
        "customer_id": "string",
        "product_id": "string",
        "product_category": "string",
        "business_line": "string",
    },
    parse_dates=["purchase_date"],
)
assert len(saved_purchases) == len(cleaned_purchases)
assert list(saved_purchases.columns) == list(cleaned_purchases.columns)

validation_summary = pd.Series({
    "source_rows": len(df),
    "eligible_paid_purchase_rows": len(product_purchases),
    "excluded_positive_gift_rows": gift_product_rows,
    "cleaned_customer_date_product_rows": len(cleaned_purchases),
    "duplicate_source_rows_combined": len(product_purchases) - len(cleaned_purchases),
    "remaining_duplicate_keys": duplicate_count,
    "customers": cleaned_purchases["customer_id"].nunique(),
    "products": cleaned_purchases["product_id"].nunique(),
})

print(f"Saved validated data to {output_path}")
validation_summary

Saved validated data to d:\programming\enterprise-product-recommendation-system\data\interim\cleaned_purchases.csv


source_rows                           113621
eligible_paid_purchase_rows            76797
excluded_positive_gift_rows             1954
cleaned_customer_date_product_rows     71916
duplicate_source_rows_combined          4881
remaining_duplicate_keys                   0
customers                               3711
products                                 927
dtype: int64

## Next step

Use `cleaned_purchases.csv` to inspect the purchase-only result and build historical scoring groups and CatBoost feature rows. When loading it again, explicitly parse `purchase_date` as a date and the identifier columns as strings. The single `quantity` column represents paid product purchases; gifts do not enter any later stage.